# STRATEGY: Aggressive Regularization + Simpler Model
Focus: Fix overfitting (CV 0.21 → Test -0.03) by reducing model complexity & increasing regularization

**Changes from ensemble:**
1. Remove engineered features (use original 445 only)
2. Reduce model complexity: max_depth 6→4, n_estimators 800→250
3. Stronger regularization: lambda 1.0→10.0, alpha 0.5→5.0
4. Better subsample/colsample: 0.7→0.8
5. Add min_child_weight: 3 (default behavior) 

**Expected:** CV ~0.19 (acceptable drop), Test R² should improve to -0.001 or better

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import gc
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("AGGRESSIVE REGULARIZATION: Reduce Overfitting")
print("="*70)

# ============== LOAD DATA ==============
print("\n[1/5] Loading data...")
train_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/train.parquet')
test_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/test.parquet')

# Compress dtypes
def compress_dtypes(df):
    for col in df.columns:
        if df[col].dtype != 'object':
            c_min, c_max = df[col].min(), df[col].max()
            if str(df[col].dtype)[:3] == 'int':
                if c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    return df

train_df = compress_dtypes(train_df)
test_df = compress_dtypes(test_df)
train_df = train_df.fillna(train_df.median())
test_df = test_df.fillna(test_df.median())

y_train = train_df['TARGET'].values.copy()
test_ids = test_df['ID'].values.copy()

X_train = train_df.drop(['ID', 'TARGET'], axis=1)
X_test = test_df.drop(['ID'], axis=1)

# Remove constant columns
const_cols = list(set(X_train.columns[X_train.nunique() <= 1]) | set(X_test.columns[X_test.nunique() <= 1]))
X_train = X_train.drop(const_cols, axis=1, errors='ignore')
X_test = X_test.drop(const_cols, axis=1, errors='ignore')

# Align columns
common_cols = sorted(list(set(X_train.columns) & set(X_test.columns)))
X_train = X_train[common_cols].copy()
X_test = X_test[common_cols].copy()

del train_df, test_df
gc.collect()
print(f"✓ Train: {X_train.shape}, Test: {X_test.shape}")

# ============== NO ENGINEERED FEATURES - USE ORIGINAL ONLY ==============
print("\n[2/5] Using ORIGINAL FEATURES ONLY (no engineering)...")
X_train_feat = X_train.copy()
X_test_feat = X_test.copy()
print(f"✓ Features: {X_train_feat.shape[1]} (original, no noise engineering)")

# ============== TRAIN WITH REGULARIZATION ==============
print("\n[3/5] Training XGBoost with AGGRESSIVE REGULARIZATION (3 seeds)...")

seeds = [42, 123, 456]
ensemble_preds = np.zeros(len(X_test_feat))
all_scores = []

# IMPROVED HYPERPARAMETERS - Focus on regularization
xgb_params = {
    'objective': 'reg:squarederror',
    'max_depth': 4,              # REDUCED from 6 (simpler trees)
    'learning_rate': 0.05,
    'subsample': 0.8,            # UP from 0.7 (more stable samples)
    'colsample_bytree': 0.8,     # UP from 0.7 (more stable features)
    'lambda': 10.0,              # UP from 1.0 (strong L2 regularization)
    'alpha': 5.0,                # UP from 0.5 (strong L1 regularization)
    'min_child_weight': 3,       # NEW (prevent overfitting on small nodes)
    'n_jobs': 4,
    'verbosity': 0
}

for seed_idx, seed in enumerate(seeds, 1):
    print(f"\n  Model {seed_idx}/3 (seed={seed}):")
    
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    fold_indices = list(kf.split(X_train_feat))
    
    test_preds = np.zeros(len(X_test_feat))
    fold_scores = []
    
    for fold_num, (train_idx, val_idx) in enumerate(fold_indices, 1):
        X_tr, X_val = X_train_feat.iloc[train_idx], X_train_feat.iloc[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        
        # REDUCED n_estimators from 800 to 250
        model = xgb.XGBRegressor(**xgb_params, n_estimators=250, random_state=seed)
        model.fit(X_tr, y_tr, verbose=False)
        
        test_preds += model.predict(X_test_feat) / 5
        r2 = r2_score(y_val, model.predict(X_val))
        fold_scores.append(r2)
        
        del model, X_tr, X_val, y_tr, y_val
        gc.collect()
    
    avg_r2 = np.mean(fold_scores)
    all_scores.append(avg_r2)
    ensemble_preds += test_preds / 3  # Average across 3 models
    print(f"    Avg R² (5-fold): {avg_r2:.6f}")

print(f"\n✓ Ensemble trained (3 models averaged)")
print(f"✓ Individual model scores: {[f'{s:.6f}' for s in all_scores]}")
print(f"✓ Ensemble Avg R²: {np.mean(all_scores):.6f}")

# ============== CREATE SUBMISSION ==============
print("\n[4/5] Creating submission...")
submission = pd.DataFrame({'ID': test_ids, 'TARGET': ensemble_preds})
print(f"✓ Submission shape: {submission.shape}")
print(f"✓ Target - Mean: {submission['TARGET'].mean():.6f}, Std: {submission['TARGET'].std():.6f}")
print(f"✓ Target - Min: {submission['TARGET'].min():.6f}, Max: {submission['TARGET'].max():.6f}")

# ============== SAVE SUBMISSION ==============
print("\n[5/5] Saving submission...")
submission.to_csv('submission_aggressive_regularization.csv', index=False)
print(f"✓ Saved: submission_aggressive_regularization.csv")

print("\n" + "="*70)
print("AGGRESSIVE REGULARIZATION - Ready for Kaggle")
print("="*70)
print(f"✓ Strategy: Reduced overfitting via regularization")
print(f"✓ Changes: Simpler model (depth 4), stronger regularization (λ=10, α=5)")
print(f"✓ Features: Original 445 only (no noise engineering)")
print(f"✓ Expected: Better test generalization (less CV→Test gap)")
print("="*70)